Bruker pip for å laste ned nødvendige bilbioteker.

<!-- Dersom du er på windows eller bruker anaconda, er det mulig at du må installere noen av de manuelt igjennom anaconda programmet. -->

In [ ]:
%pip install lonboard geopandas matplotlib pyarrow folium mapclassify scikit-learn osmnx[all]

Importerer nødvendige biblioteker for å arbeide med geodata og visualisering.

Deretter laster vi inn datasettene for tilfluktsrom og kommuner, og konverterer koordinatsystemet til UTM sone 33N (EPSG:25833) for å kunne arbeide med dem i GeoPandas.

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from pathlib import Path
from lonboard import viz

DATASET_PATH = Path('../dataset')
BUNKER_DATASET_PATH = DATASET_PATH / 'tilfluktsrom.parquet'
KOMMUNE_DATASET_PATH = DATASET_PATH / 'kommuner.parquet'

bunker_gdf = gpd.read_parquet(BUNKER_DATASET_PATH)
kommune_gdf = gpd.read_parquet(KOMMUNE_DATASET_PATH)

bunker_gdf = bunker_gdf.to_crs("EPSG:25833")
kommune_gdf = kommune_gdf.to_crs("EPSG:25833")

<p>GeoPandas og Python har problemer med å arbeide med tidsstempel objektet i datasettene, så vi må konvertere disse til en streng først.

<cite>https://stackoverflow.com/questions/49243736/how-do-i-handle-object-of-type-timestamp-is-not-json-serializable-in-python</cite>
</p>

In [ ]:
for name in ["oppdateringsdato", "datafangstdato", "datauttaksdato"]:
    if name in kommune_gdf.columns:
        kommune_gdf[name] = kommune_gdf[name].astype(str)


A

In [ ]:
m = folium.Map(location=[58.159026, 8.017441], zoom_start=12)

kommune_gdf_4326 = kommune_gdf.to_crs("EPSG:4326")
bunker_gdf_4326 = bunker_gdf.to_crs("EPSG:4326")

viz(kommune_gdf_4326)

folium.GeoJson(
    kommune_gdf,
    name="Kommuner",
    style_function=lambda _: {"color": "blue", "weight": 1, "fillOpacity": 0.05},
    tooltip=folium.GeoJsonTooltip(fields=["kommunenavn"], aliases=["Kommune:"])
).add_to(m)

bunker_layer = folium.FeatureGroup(name="Tilfluktsrom")
for _, r in bunker_gdf_4326.iterrows():
    folium.CircleMarker(
        location=[r.geom.y, r.geom.x],
        radius=3,
        color="red",
        fill=True,
        fill_opacity=0.8,
        popup=f"{r.get('adresse', 'Ukjent adresse')}<br>Plasser: {r.get('plasser', 'N/A')}"
    ).add_to(bunker_layer)

bunker_layer.add_to(m)
folium.LayerControl().add_to(m)

m

A

In [ ]:
valgt_kommune = "Oslo"
kommune_filtrert = kommune_gdf[kommune_gdf['kommunenavn'] == valgt_kommune]
kommune_filtrert.plot()

A

In [ ]:
bunker_buffer_gdf = gpd.GeoDataFrame(geometry=bunker_gdf.geometry.buffer(1000))

A

In [ ]:
m = folium.Map(location=[58.159026, 8.017441], zoom_start=12)

viz(kommune_gdf_4326)

folium.GeoJson(
    kommune_gdf_4326,
    name="Kommuner",
    style_function=lambda _: {"color": "blue", "weight": 1, "fillOpacity": 0.05},
    tooltip=folium.GeoJsonTooltip(fields=["kommunenavn"], aliases=["Kommune:"])
).add_to(m)

folium.GeoJson(
    bunker_buffer_gdf.to_crs("EPSG:4326"),
    name="1 km buffer",
    style_function=lambda _: {"color": "orange", "weight": 1, "fillOpacity": 0.06},
    tooltip=folium.GeoJsonTooltip(fields=[], aliases=[])
).add_to(m)

bunker_layer = folium.FeatureGroup(name="Tilfluktsrom")
for _, r in bunker_gdf_4326.iterrows():
    folium.CircleMarker(
        location=[r.geom.y, r.geom.x],
        radius=3,
        color="red",
        fill=True,
        fill_opacity=0.8,
        popup=f"{r.get('adresse', 'Ukjent adresse')}<br>Plasser: {r.get('plasser', 'N/A')}"
    ).add_to(bunker_layer)

bunker_layer.add_to(m)
folium.LayerControl().add_to(m)

m

A

In [ ]:
uten_dekning = kommune_filtrert.overlay(bunker_buffer_gdf, how="difference")

# Areal i km² (CRS er meterbasert: EPSG:25833)
kommune_areal_km2 = kommune_filtrert.geom.area.sum() / 1_000_000
uten_dekning_areal_km2 = uten_dekning.geom.area.sum() / 1_000_000
dekning_pct = 100 * (1 - uten_dekning_areal_km2 / kommune_areal_km2)

fig, ax = plt.subplots(figsize=(9, 9))

# Kommune grense i bakgrunnen
kommune_filtrert.boundary.plot(ax=ax, color="black", linewidth=1.2, label=f"{valgt_kommune} grense")

# Områder uten dekning
uten_dekning.plot(
    ax=ax,
    color="#f97316",
    edgecolor="#7c2d12",
    linewidth=0.8,
    alpha=0.75,
    label="Uten dekning (1 km fra tilfluktsrom)"
)

ax.set_title(
    f"Tilfluktsromdekning i {valgt_kommune}\n"
    "Fargede områder er uten dekning og mer enn 1 km fra nærmeste tilfluktsrom\n"
    f"Uten dekning: {uten_dekning_areal_km2:.1f} km² | Dekning: {dekning_pct:.1f}%",
    fontsize=13
)
ax.set_axis_off()
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

A

In [ ]:
joined = gpd.sjoin(bunker_gdf, kommune_gdf, how="left", predicate="within")
joined.head()

A

In [ ]:
bunkere_per_kommune = joined.groupby("kommunenavn").size().reset_index(name="antall_bunkere")

with pd.option_context('display.max_rows', None):
    print(bunkere_per_kommune)